# verify_book1 — Phi-4 Multimodal (HuggingFace Transformers)

Runs **microsoft/Phi-4-multimodal-instruct** locally inside Colab to answer questions from `book1.json`.

- Images are decoded from base64 embedded in the JSON
- Sub-questions are combined into one prompt per entry
- Answers written to `/content/outputs/book1/<id>.txt`
- Already-answered entries are skipped on re-run

> **Runtime:** A100 GPU (Colab Pro) strongly recommended — model is ~14B parameters.

In [ ]:
!pip install -q transformers accelerate pillow torch torchvision

In [ ]:
# HuggingFace token (needed for gated model access)
from huggingface_hub import login
from google.colab import userdata

# Store your token in Colab Secrets as HF_TOKEN
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = input("Paste your HuggingFace token: ").strip()

login(token=hf_token)
print("Logged in to HuggingFace.")

In [ ]:
# Load Phi-4 model and processor
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = "microsoft/Phi-4-multimodal-instruct"

print("Loading processor…")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Loading model (this may take a few minutes)…")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()
print("Model loaded on:", next(model.parameters()).device)

In [ ]:
# Upload book1.json
from google.colab import files
uploaded = files.upload()  # select book1.json
JSON_PATH = list(uploaded.keys())[0]
print("Using:", JSON_PATH)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
import os
OUTPUT_DIR      = "/content/outputs/book1"
MAX_NEW_TOKENS  = 1024

SYSTEM_PROMPT = (
    "You are an expert computer science and networking tutor. "
    "Answer each sub-question step by step. "
    "Label your answers clearly (e.g. (a), (b), (c)). "
    "Show your reasoning before giving the final answer."
)

# Filter: set to None to run all, or e.g. ["1", "3"] for specific ids
FILTER_IDS = None
# Filter: set to an integer to run only the last N entries
LAST_N     = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
import json, base64, io, time
from PIL import Image

def load_entries(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def build_prompt_text(entry):
    lines = [entry["question"].strip()]
    sub_qs = entry.get("sub_questions", [])
    if sub_qs:
        lines.append("")
        lines.append("Sub-questions:")
        for sq in sub_qs:
            lines.append(f"  ({sq.get('id', '')}) {sq.get('sub_question', '').strip()}")
    return "\n".join(lines)

def b64_to_pil(b64_str):
    data = base64.b64decode(b64_str)
    return Image.open(io.BytesIO(data)).convert("RGB")

def resize_to_224(img):
    """Pad to square then resize to 224x224 to avoid Phi-4 HD-tiling issues."""
    w, h   = img.size
    side   = max(w, h)
    padded = Image.new("RGB", (side, side), (255, 255, 255))
    padded.paste(img, ((side - w) // 2, (side - h) // 2))
    return padded.resize((224, 224), Image.LANCZOS)

def ask(entry):
    out_path = os.path.join(OUTPUT_DIR, f"{entry['id']}.txt")
    if os.path.exists(out_path):
        print(f"  [SKIP] #{entry['id']} — already answered")
        return

    question_text = build_prompt_text(entry)
    if not question_text.strip():
        print(f"  [SKIP] #{entry['id']} — empty question")
        return

    pil_images = []
    for img_data in entry.get("images", []):
        b64 = img_data.get("base64", "")
        if b64:
            pil_images.append(resize_to_224(b64_to_pil(b64)))

    # Build Phi-4 prompt with image placeholders
    image_tags = "".join(f"<|image_{i+1}|>" for i in range(len(pil_images)))
    phi4_prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{image_tags}{question_text}<|end|>\n"
        f"<|assistant|>\n"
    )

    inputs = processor(
        text=phi4_prompt,
        images=pil_images if pil_images else None,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = processor.decode(new_tokens, skip_special_tokens=True).strip()

    with open(out_path, "w", encoding="utf-8") as f:
        f.write(answer)
    print(f"  [OK]   #{entry['id']} → {out_path}")

print("Helpers defined.")

In [ ]:
# Load and filter entries
all_entries = load_entries(JSON_PATH)

if FILTER_IDS:
    id_set  = set(str(i) for i in FILTER_IDS)
    entries = [e for e in all_entries if str(e["id"]) in id_set]
elif LAST_N:
    entries = all_entries[-LAST_N:]
else:
    entries = all_entries

print(f"Model  : {MODEL_ID}")
print(f"Output : {OUTPUT_DIR}")
print(f"Running {len(entries)} entr{'y' if len(entries) == 1 else 'ies'}…")

In [ ]:
# Run
for entry in entries:
    try:
        ask(entry)
    except Exception as exc:
        import traceback
        print(f"  [ERR]  #{entry['id']} — {type(exc).__name__}: {exc}")
        traceback.print_exc()

print("\nDone.")

In [ ]:
# Download all output files as a zip
import shutil
from google.colab import files

zip_path = "/content/book1_phi4_answers"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
files.download(zip_path + ".zip")